In [31]:
sentence = "the capital of united states and the capital of france"
context_length = 32

In [32]:
import torch
from my_tokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")

torch.manual_seed(1)

embeddings = torch.nn.Embedding(num_embeddings=64, embedding_dim=4)

In [33]:
sentence = "the capital of united states and the capital of france"
tokens = tokenizer.encode(sentence)
tokens = torch.tensor(tokens)

meanings = embeddings(tokens)

C:\Users\Zeynep\AppData\Local\Temp\ipykernel_17300\3387454732.py:3: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [34]:
import torch

def get_rotary_position_encoding(input: torch.Tensor, base: 10000, device="cpu"):
    context_length, dimension = input.shape
    assert dimension % 2 == 0, "Dimension must be even"

    half_dimension = dimension // 2
    freqs_indices = torch.arange(0, half_dimension, device=device, dtype=torch.float32)
    freqs = 1.0 / (base ** (freqs_indices / dimension))
    position = torch.arange(0, context_length, device=device, dtype=torch.float32)

    # ✅ broadcast uyumlu hale getirme
    angles = position[:, None] * freqs[None, :]
    sin_angles = torch.sin(angles)
    cos_angles = torch.cos(angles)

    input_even = input[:, :half_dimension]
    input_odd = input[:, half_dimension:]

    input_rotated_even = input_even * cos_angles - input_odd * sin_angles
    input_rotated_odd  = input_odd  * cos_angles + input_even * sin_angles

    input_rotated = torch.empty_like(input)
    input_rotated[:, :half_dimension] = input_rotated_even
    input_rotated[:, half_dimension:] = input_rotated_odd

    return input_rotated

In [42]:
torch.manual_seed(1)
random_input = torch.randn(context_length, 4)

out=get_rotary_position_encoding(random_input, 1000, device="cpu")

In [36]:
meanings_with_rot_encoding = get_rotary_position_encoding(meanings, 1000, device="cpu")

In [37]:
import plotly.graph_objects as go
import plotly.offline

def plot_dots(sentences_data, title, dims=[0, 1, 2]):
  data = [
    go.Scatter3d(
      x=sentence_data["words"][:, dims[0]],
      y=sentence_data["words"][:, dims[1]],
      z=sentence_data["words"][:, dims[2]],
      mode="markers+text",
      marker=dict(
        size=6,
        color=sentence_data["color"],
      ),
      text=sentence_data["labels"],
      hoverinfo="text",
    ) for sentence_data in sentences_data
  ]

  layout = go.Layout(
    scene=dict(
      xaxis_title="Sertlik",
      yaxis_title="Parlaklık",
      zaxis_title="Kırmızılık",
    ),
    title=title,
  )

  fig = go.Figure(data=data, layout=layout)
  plotly.offline.iplot(fig)

In [40]:
sentences = [
    {
        "words": meanings_with_rot_encoding.detach().numpy(),
        "labels": tokenizer.tokenize(sentence),
        "color": "red",
    },
    {
        "words":meanings.detach().numpy(),
        "labels": tokenizer.tokenize(sentence),
        "color": "blue",
    }
    ]

plot_dots(sentences, "Rotary Position Encoding (RoPE) Uygulaması")